# OCPM Business Insights — Phân tích hướng đối tượng

Đây là phần phân tích **chỉ OCPM mới làm được** — không thể thực hiện bằng Traditional Process Mining vì Traditional PM không phân biệt được Offer số 1, số 2, số 3 trong cùng một Application.

| # | Câu hỏi | Insight đặc trưng OCPM |
|---|---------|------------------------|
| **Q1** | Offer thứ mấy được chấp nhận nhiều nhất? | Chiến lược Offer tối ưu |
| **Q2** | Thời gian giữa các Offer liên tiếp là bao lâu? | Bottleneck giữa các Offer |
| **Q3** | Trajectory: Offer bị Hủy (Cancelled) vs Từ chối (Refused) khác nhau thế nào? | Dự đoán kết quả sớm |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Môi trường sẵn sàng')

In [ ]:
# Tải dữ liệu
relations_df = pd.read_csv('output/bpi2017_relations.csv')
relations_df['ocel:timestamp'] = pd.to_datetime(relations_df['ocel:timestamp'], utc=True, errors='coerce')

# Dữ liệu gốc để cross-check
df_raw = pd.read_csv('../data/bpi-challenge-2017/bpi_2017_cleaned.csv')
df_raw['time:timestamp'] = pd.to_datetime(df_raw['time:timestamp'], utc=True, errors='coerce')
df_raw = df_raw.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f'✅ Relations: {len(relations_df):,} rows')
print(f'✅ Raw events: {len(df_raw):,} rows')

## Chuẩn bị: Xây dựng bảng Offer có thứ tự

Đây là bước nền tảng — gán **thứ tự (rank)** cho từng Offer trong cùng một Application.
Traditional PM **không thể làm điều này** vì nó không phân biệt các Offer.

```
Application_001:
   Offer_A  → created 10/01  → Rank = 1  (Offer đầu tiên)
   Offer_B  → created 15/01  → Rank = 2  (Offer thứ hai)
   Offer_C  → created 20/01  → Rank = 3  (Offer thứ ba)
```

In [ ]:
# Lấy các event Offer có OfferID
offer_events = df_raw[df_raw['OfferID'].notna()].copy()

# Lấy thời điểm xuất hiện đầu tiên của mỗi Offer
offer_first_seen = (
    offer_events
    .groupby(['case:concept:name', 'OfferID'])['time:timestamp']
    .min()
    .reset_index(name='offer_start_time')
)

# Gán rank: Offer nào xuất hiện sớm nhất trong Application → Rank 1
offer_first_seen['offer_rank'] = (
    offer_first_seen
    .groupby('case:concept:name')['offer_start_time']
    .rank(method='first')
    .astype(int)
)

# Lấy kết cục cuối cùng của mỗi Offer
offer_last_event = (
    offer_events
    .sort_values('time:timestamp')
    .groupby('OfferID')
    .last()[['concept:name', 'case:concept:name', 'time:timestamp']]
    .reset_index()
)
offer_last_event.columns = ['OfferID', 'final_activity', 'parent_application', 'offer_end_time']

# Phân loại kết cục
def classify_final(activity):
    if 'Accept' in str(activity):  return 'Accepted'
    elif 'Refus' in str(activity): return 'Refused'
    elif 'Cancel' in str(activity):return 'Cancelled'
    else:                           return 'Other'

offer_last_event['outcome'] = offer_last_event['final_activity'].apply(classify_final)

# Merge rank vào kết cục
offer_df = offer_first_seen.merge(
    offer_last_event[['OfferID', 'outcome', 'offer_end_time']],
    on='OfferID', how='left'
)

print(f'✅ Tổng số Offer objects: {len(offer_df):,}')
print(f'   - Offer rank 1 (đầu tiên): {(offer_df["offer_rank"]==1).sum():,}')
print(f'   - Offer rank 2:             {(offer_df["offer_rank"]==2).sum():,}')
print(f'   - Offer rank 3+:            {(offer_df["offer_rank"]>=3).sum():,}')
display(offer_df.head(5))

## Q1: Offer thứ mấy được chấp nhận nhiều nhất?

**Đây là câu hỏi KHÔNG THỂ trả lời bằng Traditional PM** vì Traditional PM gộp tất cả Offer lại thành 1 trace.

In [ ]:
# Tỷ lệ chấp nhận theo rank của Offer
rank_analysis = (
    offer_df
    .groupby('offer_rank')['outcome']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    * 100
)

# Chỉ lấy rank 1-5 (đủ dữ liệu)
rank_analysis = rank_analysis[rank_analysis.index <= 5]

# Số lượng mỗi rank
rank_counts = offer_df[offer_df['offer_rank'] <= 5]['offer_rank'].value_counts().sort_index()

print('📊 Tỷ lệ kết cục (%) theo thứ tự Offer:')
display(rank_analysis.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Stacked bar — phân phối kết cục theo rank ---
outcome_colors = {
    'Accepted': '#2ecc71',
    'Refused':  '#e74c3c',
    'Cancelled':'#f39c12',
    'Other':    '#95a5a6'
}

bottom = np.zeros(len(rank_analysis))
ranks  = rank_analysis.index.tolist()
rank_labels = [f'Offer #{r}\n(n={rank_counts.get(r, 0):,})' for r in ranks]

for outcome in ['Accepted', 'Refused', 'Cancelled', 'Other']:
    if outcome in rank_analysis.columns:
        vals = rank_analysis[outcome].values
        axes[0].bar(rank_labels, vals, bottom=bottom,
                    color=outcome_colors[outcome], label=outcome, edgecolor='white')
        # Hiển thị % nếu đủ lớn
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 5:
                axes[0].text(i, b + v/2, f'{v:.0f}%',
                             ha='center', va='center', fontsize=9,
                             color='white', fontweight='bold')
        bottom += vals

axes[0].set_title('Phân phối kết cục theo thứ tự Offer\n(OCPM: không thể thấy bằng Traditional PM)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Thứ tự Offer trong cùng Application', fontsize=11)
axes[0].set_ylabel('Tỷ lệ (%)', fontsize=11)
axes[0].legend(loc='lower right', fontsize=9)

# --- Plot 2: Tỷ lệ Accepted theo rank ---
if 'Accepted' in rank_analysis.columns:
    acc_rates = rank_analysis['Accepted'].values
    colors_bar = ['#27ae60' if v == max(acc_rates) else '#85c1e9' for v in acc_rates]
    bars = axes[1].bar(rank_labels, acc_rates, color=colors_bar, edgecolor='white', alpha=0.9)
    axes[1].set_title('Tỷ lệ chấp nhận (Acceptance Rate)\ntheo thứ tự Offer',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Thứ tự Offer trong cùng Application', fontsize=11)
    axes[1].set_ylabel('Tỷ lệ chấp nhận (%)', fontsize=11)
    axes[1].set_ylim(0, max(acc_rates) * 1.2)

    for bar, val in zip(bars, acc_rates):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

    # Đánh dấu Offer tốt nhất
    best_rank = rank_analysis['Accepted'].idxmax()
    axes[1].axhline(y=rank_analysis.loc[best_rank, 'Accepted'], color='red',
                    linestyle='--', alpha=0.5, label=f'Best: Offer #{best_rank}')
    axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('output/q1_offer_rank_acceptance.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output/q1_offer_rank_acceptance.png')

In [ ]:
# Rút ra insight
if 'Accepted' in rank_analysis.columns:
    best_rank    = rank_analysis['Accepted'].idxmax()
    best_rate    = rank_analysis.loc[best_rank, 'Accepted']
    rank1_rate   = rank_analysis.loc[1, 'Accepted'] if 1 in rank_analysis.index else 0

    print('=' * 60)
    print('📌 INSIGHT Q1: Thứ tự Offer tối ưu')
    print('=' * 60)
    print(f'  • Offer #1 (đầu tiên): Tỷ lệ chấp nhận = {rank1_rate:.1f}%')
    print(f'  • Offer #{best_rank} (tốt nhất):  Tỷ lệ chấp nhận = {best_rate:.1f}%')
    if best_rank > 1:
        print(f'\n  ✅ Kết luận: Offer thứ {best_rank} có tỷ lệ chấp nhận cao nhất!')
        print(f'     → Ngân hàng nên ưu tiên tối ưu hóa chiến lược tái offer')
        print(f'     → Không từ bỏ khách hàng sau lần từ chối đầu tiên')
    else:
        print(f'\n  ✅ Kết luận: Offer đầu tiên đã có tỷ lệ chấp nhận cao nhất')
        print(f'     → Ngân hàng nên tập trung vào chất lượng Offer đầu tiên')

## Q2: Thời gian giữa các Offer liên tiếp là bao lâu?

> **Hypothesis:** Có một khoảng chờ (bottleneck) đáng kể giữa khi một Offer kết thúc và khi ngân hàng tạo Offer mới. Đây chính là "inter-offer delay" mà Traditional PM không thể đo được.

**Đây là câu hỏi KHÔNG THỂ trả lời bằng Traditional PM** vì Traditional PM không có khái niệm "từ Offer N sang Offer N+1".

In [ ]:
# Tính thời gian giữa các Offer liên tiếp
# Sắp xếp các Offer theo Application và rank
multi_offer_apps = offer_df[offer_df['case:concept:name'].map(
    offer_df.groupby('case:concept:name')['offer_rank'].max()
) >= 2].copy()

# Tính inter-offer time: thời điểm Offer N+1 bắt đầu - thời điểm Offer N kết thúc
multi_offer_apps = multi_offer_apps.sort_values(['case:concept:name', 'offer_rank'])

multi_offer_apps['next_offer_start'] = (
    multi_offer_apps.groupby('case:concept:name')['offer_start_time'].shift(-1)
)

# Inter-offer delay = thời điểm Offer tiếp theo bắt đầu - thời điểm Offer hiện tại kết thúc
multi_offer_apps['inter_offer_hours'] = (
    (multi_offer_apps['next_offer_start'] - multi_offer_apps['offer_end_time'])
    .dt.total_seconds() / 3600
)

# Chỉ lấy các hàng có giá trị (không phải Offer cuối cùng)
transition_df = multi_offer_apps.dropna(subset=['inter_offer_hours'])
transition_df = transition_df[transition_df['inter_offer_hours'] >= 0]

# Chia theo transition: 1→2, 2→3, 3+
transition_df['transition'] = transition_df['offer_rank'].apply(
    lambda r: f'Offer #{r} → #{r+1}' if r <= 3 else f'Offer #{r} → #{r+1}'
)

print(f'📊 Số transition được phân tích: {len(transition_df):,}')
summary = transition_df.groupby('transition')['inter_offer_hours'].agg(
    count='count',
    mean_hours=lambda x: x.mean(),
    median_hours='median',
    mean_days=lambda x: x.mean() / 24
).round(2)
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Boxplot inter-offer delay theo transition ---
transitions = transition_df['transition'].unique()
transitions_sorted = sorted(transitions)

data_for_box = [
    transition_df.loc[
        (transition_df['transition'] == t) &
        (transition_df['inter_offer_hours'] < transition_df['inter_offer_hours'].quantile(0.95)),
        'inter_offer_hours'
    ].values
    for t in transitions_sorted
]

bp = axes[0].boxplot(data_for_box, labels=transitions_sorted, patch_artist=True,
                      showfliers=False, medianprops={'color': 'red', 'linewidth': 2})
colors_box = ['#3498db', '#e67e22', '#9b59b6']
for patch, color in zip(bp['boxes'], colors_box[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_title('Inter-Offer Delay: Thời gian chờ giữa\ncác Offer liên tiếp (Bottleneck OCPM)',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Transition')
axes[0].set_ylabel('Thời gian chờ (giờ)')
axes[0].tick_params(axis='x', rotation=15)

# Thêm annotation median
for i, t in enumerate(transitions_sorted):
    med = transition_df[transition_df['transition'] == t]['inter_offer_hours'].median()
    axes[0].text(i + 1, med + 1, f'Med: {med:.1f}h\n({med/24:.1f}d)',
                  ha='center', fontsize=9, color='darkred', fontweight='bold')

# --- Plot 2: Mean inter-offer delay (days) ---
mean_days = [
    transition_df[transition_df['transition'] == t]['inter_offer_hours'].mean() / 24
    for t in transitions_sorted
]

bars = axes[1].bar(transitions_sorted, mean_days,
                    color=colors_box[:len(transitions_sorted)], edgecolor='white', alpha=0.85)
axes[1].set_title('Inter-Offer Delay Trung bình (ngày)\n"Ngân hàng mất bao lâu để tạo Offer tiếp theo?"',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Transition')
axes[1].set_ylabel('Số ngày trung bình')
axes[1].tick_params(axis='x', rotation=15)

for bar, val in zip(bars, mean_days):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                  f'{val:.1f} ngày', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('output/q2_inter_offer_delay.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output/q2_inter_offer_delay.png')

In [ ]:
# So sánh: inter-offer delay vs customer wait (từ phân tích trước)
customer_wait_mean = 10.06  # ngày, từ bank_vs_customer_wait_analysis
bank_wait_mean = 2.35       # ngày, từ bank_vs_customer_wait_analysis

first_transition = transitions_sorted[0] if transitions_sorted else None
if first_transition:
    inter_offer_mean_days = transition_df[
        transition_df['transition'] == first_transition
    ]['inter_offer_hours'].mean() / 24

    print('=' * 65)
    print('📌 INSIGHT Q2: Inter-Offer Bottleneck')
    print('=' * 65)
    print(f'  • Customer wait (trung bình):      {customer_wait_mean:.2f} ngày')
    print(f'  • Bank review wait (trung bình):   {bank_wait_mean:.2f} ngày')
    print(f'  • Inter-offer delay ({first_transition}): {inter_offer_mean_days:.2f} ngày')
    print()
    print(f'  ⚠️  Kết luận: Khi một Offer bị từ chối/hủy, ngân hàng cần')
    print(f'     trung bình {inter_offer_mean_days:.1f} ngày để tạo Offer tiếp theo!')
    print(f'     Đây là bottleneck ẩn mà Traditional PM không nhìn thấy.')
    print(f'\n  💡 Đề xuất: Tự động hóa việc tạo Offer mới khi Offer cũ bị từ chối')
    print(f'     → Có thể rút ngắn đáng kể tổng thời gian xử lý Application')

## Q3: Trajectory phân tích — HỦY (Cancelled) khác với TỪ CHỐI (Refused) như thế nào?

> **Bối cảnh:** Trong dữ liệu BPI 2017:
- `O_Refused`: Khách hàng TỪ CHỐI Offer (dẫn đến hồ sơ bị đóng).
- `O_Cancelled`: Offer bị HỦY (thường do ngân hàng hoặc thỏa thuận lại để tạo Offer mới).

**Đây là phân tích Trajectory thuần OCPM**: Sự khác biệt sinh tử giữa việc Offer đầu tiên bị Cancelled so với bị Refused.

In [ ]:
# Xây dựng trajectory chuỗi kết cục cho từng Application
app_trajectories = (
    offer_df
    .sort_values(['case:concept:name', 'offer_rank'])
    .groupby('case:concept:name')
    .apply(lambda g: list(g['outcome']))
    .reset_index(name='trajectory')
)

# Kết quả cuối cùng của Application (có được chấp nhận không?)
# Trong BPI 2017, A_Accepted xảy ra trước khi tạo Offer. Kết cục thành công thực sự là A_Pending (chờ giải ngân)
accepted_cases = df_raw[df_raw['concept:name'] == 'A_Pending']['case:concept:name'].unique()
app_trajectories['app_accepted'] = app_trajectories['case:concept:name'].isin(accepted_cases)

# Encode trajectory thành chuỗi ngắn gọn
def encode_trajectory(traj):
    short = {'Accepted': 'A', 'Refused': 'R', 'Cancelled': 'C', 'Other': '?'}
    return ' → '.join([short.get(t, '?') for t in traj[:4]])  # max 4 offers

app_trajectories['trajectory_str'] = app_trajectories['trajectory'].apply(encode_trajectory)

print(f'📊 Tổng số Applications có Offer: {len(app_trajectories):,}')
print(f'   Chú thích: A=Accepted, R=Refused, C=Cancelled')
print()

# Top 10 trajectory phổ biến nhất
top_trajectories = (
    app_trajectories.groupby('trajectory_str')
    .agg(count=('case:concept:name', 'count'),
         acceptance_rate=('app_accepted', 'mean'))
    .sort_values('count', ascending=False)
    .head(10)
    .reset_index()
)
top_trajectories['acceptance_pct'] = (top_trajectories['acceptance_rate'] * 100).round(1)
top_trajectories['count_pct'] = (top_trajectories['count'] / len(app_trajectories) * 100).round(1)

print('Top 10 Offer Trajectories phổ biến nhất:')
display(top_trajectories[['trajectory_str', 'count', 'count_pct', 'acceptance_pct']])

In [ ]:
# Phân tích: Offer đầu bị Refused/Cancelled → xác suất Application được chấp nhận?

def get_first_offer_outcome(traj):
    return traj[0] if traj else 'Unknown'

def get_second_offer_outcome(traj):
    return traj[1] if len(traj) > 1 else 'No 2nd Offer'

app_trajectories['first_offer']  = app_trajectories['trajectory'].apply(get_first_offer_outcome)
app_trajectories['second_offer'] = app_trajectories['trajectory'].apply(get_second_offer_outcome)
app_trajectories['num_offers']   = app_trajectories['trajectory'].apply(len)

# Tỷ lệ Application được chấp nhận theo kết cục Offer #1
recovery_by_first = (
    app_trajectories.groupby('first_offer')
    .agg(count=('case:concept:name', 'count'),
         app_acceptance=('app_accepted', 'mean'))
    .reset_index()
)
recovery_by_first['acceptance_pct'] = (recovery_by_first['app_acceptance'] * 100).round(1)

print('📊 Xác suất Application được chấp nhận theo kết cục Offer #1:')
display(recovery_by_first)

In [ ]:
# Conditional analysis: Khi Offer #1 bị HỦY (Cancelled), Offer #2 cứu vãn được bao nhiêu?
cancelled_first = app_trajectories[app_trajectories['first_offer'] == 'Cancelled'].copy()

recovery_conditional = (
    cancelled_first.groupby('second_offer')
    .agg(count=('case:concept:name', 'count'),
         app_acceptance=('app_accepted', 'mean'))
    .reset_index()
)
recovery_conditional['acceptance_pct'] = (recovery_conditional['app_acceptance'] * 100).round(1)
recovery_conditional = recovery_conditional.sort_values('count', ascending=False)

print(f'📊 Khi Offer #1 bị HỦY (Cancelled) (n={len(cancelled_first):,}):')
print('   Xác suất Application được chấp nhận ở Offer #2 là bao nhiêu?')
display(recovery_conditional)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Recovery rate theo kết cục Offer #1 ---
outcomes_order = ['Accepted', 'Refused', 'Cancelled', 'Other']
rf1 = recovery_by_first.set_index('first_offer').reindex(
    [o for o in outcomes_order if o in recovery_by_first['first_offer'].values]
)

colors_outcome = {'Accepted': '#2ecc71', 'Refused': '#e74c3c',
                   'Cancelled': '#f39c12', 'Other': '#95a5a6'}
bar_colors = [colors_outcome.get(o, '#95a5a6') for o in rf1.index]

bars1 = axes[0].bar(rf1.index, rf1['acceptance_pct'].values,
                     color=bar_colors, edgecolor='white', alpha=0.85)
axes[0].set_title('Xác suất Application được chấp nhận\ntheo kết cục Offer đầu tiên',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Kết cục Offer #1', fontsize=11)
axes[0].set_ylabel('Xác suất Application được chấp nhận (%)', fontsize=11)
axes[0].set_ylim(0, 110)

for bar, (idx, row) in zip(bars1, rf1.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                  f"{row['acceptance_pct']:.1f}%\n(n={int(row['count']):,})",
                  ha='center', fontsize=10, fontweight='bold')

# --- Plot 2: Conditional — Nếu Offer #1 Cancelled, Offer #2 ảnh hưởng gì? ---
if len(recovery_conditional) > 0:
    rc = recovery_conditional[recovery_conditional['second_offer'] != 'Unknown'].head(5)
    bar_colors2 = [colors_outcome.get(o, '#95a5a6') for o in rc['second_offer']]

    bars2 = axes[1].bar(rc['second_offer'], rc['acceptance_pct'],
                         color=bar_colors2, edgecolor='white', alpha=0.85)
    axes[1].set_title('Conditional: Offer #1 bị HỦY (Cancelled)\nXác suất Application được chấp nhận nhờ Offer #2',
                       fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Kết cục Offer #2', fontsize=11)
    axes[1].set_ylabel('Xác suất Application được chấp nhận (%)', fontsize=11)
    axes[1].set_ylim(0, 110)

    for bar, (_, row) in zip(bars2, rc.iterrows()):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                      f"{row['acceptance_pct']:.1f}%\n(n={int(row['count']):,})",
                      ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('output/q3_offer_trajectory_recovery.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output/q3_offer_trajectory_recovery.png')

In [ ]:
# Rút ra insight Q3
if 'Refused' in rf1.index and 'Cancelled' in rf1.index:
    acc_if_first_refused  = rf1.loc['Refused', 'acceptance_pct']
    acc_if_first_cancelled  = rf1.loc['Cancelled', 'acceptance_pct']

    print('=' * 65)
    print('📌 INSIGHT Q3: Trajectory — TỪ CHỐI vs HỦY BỎ')
    print('=' * 65)
    print(f'  • Nếu Offer #1 bị khách hàng TỪ CHỐI (Refused)  → Application accepted: {acc_if_first_refused:.1f}%')
    print(f'  • Nếu Offer #1 bị HỦY BỎ (Cancelled)            → Application accepted: {acc_if_first_cancelled:.1f}%')
    print()

    if not recovery_conditional.empty:
        # Tỷ lệ khi Offer 2 được Accepted
        if 'Accepted' in recovery_conditional['second_offer'].values:
            acc_offer2 = recovery_conditional[recovery_conditional['second_offer'] == 'Accepted'].iloc[0]
            print(f'  • Khi Offer #1 bị HỦY + Offer #2 được CHẤP NHẬN')
            print(f'    → Xác suất Application thành công phục hồi lên tới: {acc_offer2["acceptance_pct"]:.1f}%')

    print()
    print(f'  💡 Kết luận chiến lược:')
    print(f'     1. O_Refused thực sự là DẤU CHẤM HẾT cho Application ({acc_if_first_refused:.1f}% thành công).')
    print(f'     2. Tuy nhiên, O_Cancelled mang lại cơ hội thứ hai rất lớn ({acc_if_first_cancelled:.1f}% thành công).')
    print(f'     → Ngân hàng cần khuyến khích nhân viên chủ động "Cancel" các Offer không phù hợp')
    print(f'       để tạo Offer mới, TRƯỚC KHI khách hàng kịp bấm nút "Refuse"!')

## Tổng kết: OCPM vs Traditional PM

In [ ]:
print('=' * 70)
print('🏆 TỔNG KẾT: GIÁ TRỊ CỦA OCPM vs TRADITIONAL PROCESS MINING')
print('=' * 70)

print('''
┌─────────────────────────────────────────────────────────────────────┐
│                    OCPM-exclusive Insights (BPI 2017)               │
├──────────┬──────────────────────────────────────────────────────────┤
│ Q1       │ Offer thứ mấy được chấp nhận nhiều nhất?                 │
│          │ → Tối ưu hóa chiến lược offer của ngân hàng              │
├──────────┼──────────────────────────────────────────────────────────┤
│ Q2       │ Inter-offer delay: bottleneck ẩn giữa các Offer          │
│          │ → Traditional PM hoàn toàn không nhìn thấy!              │
├──────────┼──────────────────────────────────────────────────────────┤
│ Q3       │ Trajectory: Sự khác biệt sinh tử giữa Refused & Cancelled│
│          │ → Chủ động Cancel offer cũ trước khi khách Refuse!       │
├──────────┴──────────────────────────────────────────────────────────┤
│ Khuyến nghị cho ngân hàng:                                          │
│ 1. Chủ động Cancel Offer không phù hợp để đàm phán lại.             │
│ 2. Giảm thiểu "inter-offer delay" (thời gian chờ giữa 2 Offer).     │
│ 3. Hạn chế để khách hàng bấm Refuse (tỷ lệ phục hồi 0%).            │
└─────────────────────────────────────────────────────────────────────┘
''')

print('📊 Output files:')
print('  • output/q1_offer_rank_acceptance.png')
print('  • output/q2_inter_offer_delay.png')
print('  • output/q3_offer_trajectory_recovery.png')